# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

metadata = dataset.metadata
print(f"{getattr(metadata, 'name', 'Unnamed Dataset')}: {getattr(metadata, 'description', 'No description available.')}")

## 2. Data Overview
Review available record sets (tables), their fields (columns), and their `@id`.

All entities are referenced **by their `@id`** as per the Croissant specification.

In [ ]:
# List record sets and fields using their @id
from pprint import pprint

record_sets = dataset.record_sets

print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    print(f"  name: {rs.get('name', '')}")
    print(f"  description: {rs.get('description', '')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields:")
    for field in fields:
        if isinstance(field, dict):
            fid = field.get('@id', None)
            fname = field.get('name', '')
            fdesc = field.get('description', '')
        else:
            fid = field
            fname = ''
            fdesc = ''
        print(f"    - {fid}: {fname}{' | '+fdesc if fdesc else ''}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the RecordSet and Field `@id`s from the overview.

We'll load all main record sets returned above.

In [ ]:
# Extract data from each record set using @id
dataframes = {}
all_record_set_ids = [rs['@id'] for rs in record_sets]
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet: {record_set_id}, shape: {df.shape}")
        print("Columns (by field @id): ", df.columns.tolist())
        display(df.head(2))
    else:
        print(f"No records loaded for {record_set_id}")


## 4. Exploratory Data Analysis (EDA)
Let's select one populated record set (usually the main data table), choose a numeric field, and perform filtering, normalization, and grouping operations.

We'll use the actual `@id` of the record set and fields as referenced above. Modify the variable assignments below as needed for the field you want to explore.

In [ ]:
# For this dataset, let's pick the first record set with data
selected_record_set_id = next(iter(dataframes.keys()))  # Pick the first non-empty RecordSet
df = dataframes[selected_record_set_id]

# List numeric column candidates by simple dtype inspection
numeric_cols = df.select_dtypes(include=['number', 'float', 'int']).columns
print(f"Numeric fields found (by @id): {list(numeric_cols)}")

# If found, pick first numeric column for demo; otherwise, use fallback
if len(numeric_cols) > 0:
    numeric_field_id = numeric_cols[0]
else:
    numeric_field_id = df.columns[0]  # fallback

# EDA: Filtering, normalization, grouping
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field (if present)
cat_cols = df.select_dtypes(include=['object']).columns
group_field = None
for col in cat_cols:
    # Choose a group field with reasonable number of unique values
    if df[col].nunique() < 20 and col != numeric_field_id:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field} (@id):")
    display(grouped_df)
else:
    print("No suitable group field (categorical) found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot the selected numeric field distribution and, if available, its relationship to the chosen categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If grouping field was found, barplot the group means
if 'grouped_df' in locals() and group_field is not None:
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
    plt.title(f"Average {numeric_field_id} by {group_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

- We demonstrated how to explore the FAIR^2 dataset metadata, inspect its schema by `@id`, and load its records for analysis using the `mlcroissant` Python library.
- Demonstrated filtering and normalization on a chosen numeric field, and how to group data by relevant categorical field if present.
- Visualized core data distributions to begin further investigation.

Further analysis can focus on domain-specific queries—such as investigating protein abundances, coverage percentages, modification frequencies, or correlating specific sample attributes with protein characteristics.

**Remember:** All dataset entities are referenced by `@id`. For deeper exploration, consult the Croissant schema for field/column details.